In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, ConcatDataset
import requests
import zipfile

In [3]:
BACKEND_URL = " https://regain-agile-army.ngrok-free.dev"
API_KEY = "kalhara_secure_api_2026"

GDRIVE_SAVE_PATH = "/content/drive/MyDrive/krishi_ai_models/best_plant_model.pth"
LOCAL_MODEL_PATH = "best_plant_model.pth"

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(os.path.dirname(GDRIVE_SAVE_PATH), exist_ok=True)
    print("✅ Google Drive mounted. Model will be saved to:", GDRIVE_SAVE_PATH)
    USE_GDRIVE = True
except Exception as e:
    print(f"⚠️  Google Drive not mounted: {e}")
    print("    Model will only be saved locally (lost when session ends).")
    USE_GDRIVE = False

Mounted at /content/drive
✅ Google Drive mounted. Model will be saved to: /content/drive/MyDrive/krishi_ai_models/best_plant_model.pth


In [5]:
live_data_dir = './live_data'
headers = {"x-api-key": API_KEY}

if BACKEND_URL != "http://YOUR_NGROK_URL/api":
    print(f"Connecting to backend at {BACKEND_URL} ...")
    try:
        response = requests.get(f"{BACKEND_URL}/export-data", headers=headers, timeout=30)
        if response.status_code == 200:
            with open("verified_data.zip", "wb") as f:
                f.write(response.content)
            with zipfile.ZipFile("verified_data.zip", 'r') as zip_ref:
                zip_ref.extractall(live_data_dir)
            print("✅ Downloaded user-verified data from backend!")
        elif response.status_code == 404:
            print("ℹ️  No verified data on backend yet — training with Kaggle data only.")
        else:
            print(f"⚠️  Backend returned {response.status_code}. Training with Kaggle data only.")
    except Exception as e:
        print(f"⚠️  Could not connect to backend: {e}\n    Training with Kaggle data only.")
else:
    print("ℹ️  No backend URL set. Skipping live data download.")

Connecting to backend at  https://regain-agile-army.ngrok-free.dev ...
ℹ️  No verified data on backend yet — training with Kaggle data only.


In [6]:
class PlantDiseaseClassifier(nn.Module):
    def __init__(self, num_classes=38):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()

        self.fc1 = nn.Linear(in_features, 512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(512, 128)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = self.fc1(x);  x = self.relu1(x);  x = self.drop1(x)
        x = self.fc2(x);  x = self.relu2(x);  x = self.drop2(x)
        x = self.fc3(x)
        return x

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [8]:
print("Downloading Plant Disease Dataset from Kaggle...")
try:
    import kagglehub
    dataset_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
    print(f"✅ Dataset downloaded to: {dataset_path}")
except ImportError:
    raise RuntimeError(
        "❌ kagglehub not installed!\n"
        "Run this in a Colab cell first:\n"
        "  !pip install kagglehub\n"
        "Then restart the runtime and run again."
    )
except Exception as e:
    raise RuntimeError(f"❌ Failed to download dataset: {e}")


Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
✅ Dataset downloaded to: /kaggle/input/new-plant-diseases-dataset


In [9]:
base = dataset_path
for root, dirs, files in os.walk(base):
    if "train" in dirs and "valid" in dirs:
        train_path = os.path.join(root, "train")
        val_path   = os.path.join(root, "valid")
        break
else:
    # Fallback: search one level deeper
    augmented = os.path.join(base, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)")
    train_path = os.path.join(augmented, "train")
    val_path   = os.path.join(augmented, "valid")

print(f"Train path: {train_path}")
print(f"Valid path: {val_path}")
assert os.path.exists(train_path), f"Train path not found: {train_path}"
assert os.path.exists(val_path),   f"Val path not found: {val_path}"

train_dataset = datasets.ImageFolder(root=train_path, transform=transform)
val_dataset   = datasets.ImageFolder(root=val_path,   transform=transform)
class_names   = train_dataset.classes
print(f"✅ Found {len(class_names)} classes, {len(train_dataset)} training images.")


Train path: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
Valid path: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid
✅ Found 38 classes, 70295 training images.


In [10]:
try:
    if os.path.exists(live_data_dir) and len(os.listdir(live_data_dir)) > 0:
        live_dataset = datasets.ImageFolder(live_data_dir, transform=transform)
        full_train_dataset = ConcatDataset([train_dataset, live_dataset])
        print(f"✅ Merged Kaggle data + {len(live_dataset)} backend verified images!")
    else:
        full_train_dataset = train_dataset
        print("ℹ️  No live data to merge. Training on Kaggle data only.")
except Exception as e:
    print(f"⚠️  Could not load live data: {e}")
    full_train_dataset = train_dataset

train_loader = DataLoader(full_train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,         batch_size=32, shuffle=False, num_workers=2)

# Print class names so you can copy them into backend/app/config.py
print("\n📋 CLASS NAMES (copy these into backend/app/config.py → CLASS_NAMES):\n")
print(class_names)

ℹ️  No live data to merge. Training on Kaggle data only.

📋 CLASS NAMES (copy these into backend/app/config.py → CLASS_NAMES):

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_bl

In [ ]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Training on: {device}")
if device.type == "cpu":
    print("⚠️  WARNING: No GPU detected! Training will be very slow.")
    print("   Go to Runtime > Change runtime type > GPU (T4)")

model     = PlantDiseaseClassifier(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs       = 50
patience     = 5
best_val_loss = float('inf')
counter      = 0

for epoch in range(epochs):
    # --- Training phase ---
    model.train()
    train_loss = train_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * images.size(0)
        _, predicted   = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()

    # --- Validation phase ---
    model.eval()
    val_loss = val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss     = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss   = val_loss   / len(val_loader.dataset)
    train_acc      = train_correct / len(train_loader.dataset)
    val_acc        = val_correct   / len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # ✅ Save locally
        torch.save(model.state_dict(), LOCAL_MODEL_PATH)
        # ✅ FIXED: Also save to Google Drive so model persists after session
        if USE_GDRIVE:
            torch.save(model.state_dict(), GDRIVE_SAVE_PATH)
            print(f"  ✅ Best model saved locally + to Google Drive!")
        else:
            print(f"  ✅ Best model saved locally!")
        counter = 0
    else:
        counter += 1
        print(f"  ⚠️  No improvement. EarlyStopping: {counter}/{patience}")
        if counter >= patience:
            print("  🛑 Early stopping triggered.")
            break

    print('-' * 60)

print(f"\n🏆 Training complete! Best val loss: {best_val_loss:.4f}")


🖥️  Training on: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 218MB/s]


In [ ]:
# Store metrics
history['train_loss'].append(avg_train_loss)
history['val_loss'].append(avg_val_loss)
history['train_acc'].append(train_acc)
history['val_acc'].append(val_acc)

In [ ]:
history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': []
}

In [ ]:
if BACKEND_URL != "https://regain-agile-army.ngrok-free.dev":
    print("\nUploading model to backend...")
    try:
        with open(LOCAL_MODEL_PATH, 'rb') as f:
            files = {'file': ('latest_model.pth', f, 'application/octet-stream')}
            res = requests.post(
                f"{BACKEND_URL}/upload-model",
                headers=headers,
                files=files,
                timeout=120
            )
        if res.status_code == 200:
            print("✅ Model uploaded to backend successfully!")
            print("   Backend response:", res.json())
        else:
            print(f"❌ Upload failed: {res.status_code} - {res.text}")
            print(f"   You can manually upload the model file: {LOCAL_MODEL_PATH}")
            if USE_GDRIVE:
                print(f"   Or from Google Drive: {GDRIVE_SAVE_PATH}")
    except Exception as e:
        print(f"❌ Upload failed: {e}")
        print(f"   Manually download the model from: {LOCAL_MODEL_PATH}")
        if USE_GDRIVE:
            print(f"   Or from Google Drive: {GDRIVE_SAVE_PATH}")
else:
    print("\nℹ️  No backend URL set. Model NOT uploaded to backend.")
    print(f"   Download the model from Google Drive: {GDRIVE_SAVE_PATH}")
    print("   Then upload it manually via the backend /api/upload-model endpoint.")

### Training History Visualizations

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Training Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("\n🔍 Quick local inference test...")
try:
    import glob
    from PIL import Image
    import io

    sample_images = (
        glob.glob(os.path.join(val_path, "*", "*.jpg")) +
        glob.glob(os.path.join(val_path, "*", "*.JPG")) +
        glob.glob(os.path.join(val_path, "*", "*.png"))
    )
    if sample_images:
        test_img = Image.open(sample_images[0]).convert("RGB")
        tensor   = transform(test_img).unsqueeze(0).to(device)
        model.eval()
        with torch.no_grad():
            out  = model(tensor)
            prob = torch.nn.functional.softmax(out, dim=1)
            idx  = prob.argmax().item()
        print(f"✅ Test image: {os.path.basename(sample_images[0])}")
        print(f"   Predicted: {class_names[idx]} ({prob[0][idx].item()*100:.1f}% confidence)")
    else:
        print("No sample images found for testing.")
except Exception as e:
    print(f"Test failed: {e}")